In [1]:
# uv add langchain-community pdfplumber
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [2]:
# uv add langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [4]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3:latest",
)

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(texts, embeddings) # in-memory

In [11]:
query = "경조사 결혼하면 휴가는 몇일이야?"

In [12]:
res = vector_store.similarity_search(query, k=3)

In [13]:
res

[Document(id='fc02c2cb-0316-4cc3-82c6-bb8b3ac754cd', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 11, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='\uf0b7 소멸: 당해 연도 12 월 31 일까지 미사용 시 자동 소멸 (이월 불가).\n5.4 경조사 지원 기준 (Family Support Details)\n기쁨과 슬픔을 함께 나누는 테크노빌드의 경조 지원입니다.\n구분 대상 휴가 (일수) 경조금 (만원) 화환/조화\n결혼 본인 5 일 100 + 화환 지원\n자녀 1 일 50 + 화환 지원\n30 -\n형제/자매 1 일\n30 -\n회갑/칠순 본인/배우자 부모 1 일\n출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니\n배우자 10 일 (유급) 출산 축하금 50 과일 바구니\n-\n사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기\n부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기\n30\n조부모/외조부모 3 일 조화\n30\n형제/자매 3 일 조화'),
 Document(id='3845604a-9ae3-4931-b530-7c63ea1632b9', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employ

In [14]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.

- 컨텍스트: {context}
- 질문: {question}"""
)

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.llms import OllamaLLM

model = OllamaLLM(model="gemma3:4b")

chain = prompt | model | StrOutputParser()

In [17]:
result = chain.invoke({
    "context": res,
    "question": query
})

In [18]:
result

'결혼 시 5일의 휴가가 제공됩니다.'

---

In [22]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0,
    # other params...
)

In [19]:
system_prompt="""당신은 임직원 통합 가이드북 정보를 제공하는 인사 담당자입니다.

1. 정보가 필요할 경우 검색 도구를 사용하여 확인하세요.
2. 일반적이거나 이미 학습된 내용은 스스로 답변을 생성하세요.
3. 임직원 통합 가이드북 정보의 검색이 필요한 경우에는 도구를 사용하세요.
"""

In [ ]:
query = "경조사 중에서 결혼하면 휴가는 몇일이고, 얼마를 받을 수 있을까?"

In [27]:
res_str = "\n\n".join([r.page_content for r in res])

In [28]:
print(res_str)

 소멸: 당해 연도 12 월 31 일까지 미사용 시 자동 소멸 (이월 불가).
5.4 경조사 지원 기준 (Family Support Details)
기쁨과 슬픔을 함께 나누는 테크노빌드의 경조 지원입니다.
구분 대상 휴가 (일수) 경조금 (만원) 화환/조화
결혼 본인 5 일 100 + 화환 지원
자녀 1 일 50 + 화환 지원
30 -
형제/자매 1 일
30 -
회갑/칠순 본인/배우자 부모 1 일
출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니
배우자 10 일 (유급) 출산 축하금 50 과일 바구니
-
사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기
부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기
30
조부모/외조부모 3 일 조화
30
형제/자매 3 일 조화

300 만원 + 웰니스 포인트 200 만P + 부부 동반 정밀 검진권.
[시나리오 2: 중기 근속자] - 대상: 2020 년 7 월 1 일 입사한 T3 직급 직원 - 현재 시점:
2026 년 1 월 1 일 (근속 약 5.5 년) - 발생 혜택: 1. 기본 연차 15 일 + 가산 연차 2 일 (3,
5 주년 가산) = 총 17 일 발생. 2. 5 주년 리프레시 휴가 10 일 (만 5 년 시점인 2025.07 에
이미 발생했으나 미사용분 이월 가능 여부 확인 필요). 3. 휴가비 100 만원 + 웰니스 포인트
120 만P.
[문서 끝] 본 가이드북의 내용은 대한민국 근로기준법 및 관련 법령, 회사 취업규칙의
변경에 따라 매년 갱신될 수 있습니다.

o 1 년 이상: 15 일 발생 + 매 2 년마다 1 일 가산 (최대 25 일).
 사용 촉진: 연차 사용 촉진 제도 시행 (미사용 수당 지급 원칙이나, 7 월/10 월 촉진
통보 후 미사용 시 소멸).
B. 리프레시 휴가 (Refres-T)
장기 근속자의 재충전을 위한 특별 휴가입니다.


In [29]:
messages = [
    (
        "system",
        system_prompt,
    ),
    ("human", query + "\n\n" + res_str),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='경조사 휴가 및 경조금 관련 정보는 다음과 같습니다.\n\n**1. 결혼 관련 경조금 및 휴가**\n\n*   **결혼 본인:** 5일 휴가 + 100만원 경조금 + 화환 지원\n*   **자녀:** 1일 휴가 + 50만원 경조금 + 화환 지원\n*   **형제/자매:** 1일 휴가\n\n**2. 기타 경조금 및 휴가**\n\n*   **회갑/칠순 본인/배우자 부모:** 1일 휴가\n*   **출산 본인:** 90일 출산휴가 + 50만원 출산 축하금 + 과일 바구니\n*   **배우자:** 10일 유급 출산축하금 + 50만원 출산 축하금 + 과일 바구니\n*   **사망 본인/배우자:** 500만원 + 장례용품 3단 조화 + 근조기\n*   **부모/배우자 부모:** 5일 휴가 + 100만원 경조금 + 장례용품 3단 조화 + 근조기\n*   **조부모/외조부모:** 3일 조화\n\n**3. 근속 연차 휴가**\n\n*   **1년 이상 근속:** 15일 기본 연차 + 매 2년마다 1일 가산 (최대 25일)\n*   **2026년 시점 (근속 5.5년):** 17일 (기본 연차 15일 + 가산 연차 2일)\n\n**4. 추가 혜택 (중기 근속자)**\n\n*   **5년차 리프레시 휴가:** 10일 (이미 발생했으나 미사용분 이월 가능 여부 확인 필요)\n*   **휴가비:** 100만원\n*   **웰니스 포인트:** 120만 P\n\n**주의사항:**\n\n*   경조사 휴가 및 경조금은 당해 연도 12월 31일까지 미사용 시 자동 소멸됩니다.\n*   연차 사용 촉진 제도를 확인하여 미사용 수당 지급 여부를 확인해야 합니다.\n*   본 가이드북 내용은 대한민국 근로기준법 및 관련 법령, 회사 취업규칙의 변경에 따라 매년 갱신될 수 있습니다.\n\n더 궁금한 점이 있으시면 언제든지 질문해주세요.', additional_kwargs={}, response_metadata={'model': 'gemma3:4b', 'create

In [31]:
print(ai_msg.content)

경조사 휴가 및 경조금 관련 정보는 다음과 같습니다.

**1. 결혼 관련 경조금 및 휴가**

*   **결혼 본인:** 5일 휴가 + 100만원 경조금 + 화환 지원
*   **자녀:** 1일 휴가 + 50만원 경조금 + 화환 지원
*   **형제/자매:** 1일 휴가

**2. 기타 경조금 및 휴가**

*   **회갑/칠순 본인/배우자 부모:** 1일 휴가
*   **출산 본인:** 90일 출산휴가 + 50만원 출산 축하금 + 과일 바구니
*   **배우자:** 10일 유급 출산축하금 + 50만원 출산 축하금 + 과일 바구니
*   **사망 본인/배우자:** 500만원 + 장례용품 3단 조화 + 근조기
*   **부모/배우자 부모:** 5일 휴가 + 100만원 경조금 + 장례용품 3단 조화 + 근조기
*   **조부모/외조부모:** 3일 조화

**3. 근속 연차 휴가**

*   **1년 이상 근속:** 15일 기본 연차 + 매 2년마다 1일 가산 (최대 25일)
*   **2026년 시점 (근속 5.5년):** 17일 (기본 연차 15일 + 가산 연차 2일)

**4. 추가 혜택 (중기 근속자)**

*   **5년차 리프레시 휴가:** 10일 (이미 발생했으나 미사용분 이월 가능 여부 확인 필요)
*   **휴가비:** 100만원
*   **웰니스 포인트:** 120만 P

**주의사항:**

*   경조사 휴가 및 경조금은 당해 연도 12월 31일까지 미사용 시 자동 소멸됩니다.
*   연차 사용 촉진 제도를 확인하여 미사용 수당 지급 여부를 확인해야 합니다.
*   본 가이드북 내용은 대한민국 근로기준법 및 관련 법령, 회사 취업규칙의 변경에 따라 매년 갱신될 수 있습니다.

더 궁금한 점이 있으시면 언제든지 질문해주세요.
